# 98th Academy Awards Prediction Pipeline

**Ceremony**: March 15, 2026  
**Four independent methods**: Stats (calibrated season weights) · ML (elastic-net / GBT with per-category selection) · LLM (independent web research) · Gold Derby (crowd odds)

---

## Run Order
1. (Re-)run any method scripts if needed — see Cell 2
2. Adjust ensemble weights in Cell 6
3. Re-run all cells to recompute final predictions

In [1]:
# Cell 1: Imports and setup
import json
import os
import sys
import numpy as np
import pandas as pd

# Repo root is the directory containing this notebook
REPO_ROOT = os.path.dirname(os.path.abspath('pipeline.ipynb'))
if not os.path.exists(os.path.join(REPO_ROOT, 'src')):
    # Fallback: assume cwd is repo root
    REPO_ROOT = os.getcwd()

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print(f'Working directory: {os.getcwd()}')

# Load predictions
def load_json(path):
    with open(path) as f:
        return json.load(f)

stats_preds   = load_json('results/stats/predictions.json')
ml_preds      = load_json('results/ml_model/predictions.json')
llm_preds     = load_json('results/llm_analysis/predictions.json')
gd_preds      = load_json('results/goldderby/predictions.json')
stats_valid   = load_json('results/stats/validation.json')
ml_valid      = load_json('results/ml_model/validation.json')
gd_valid      = load_json('results/goldderby/validation.json')
nominees_raw  = load_json('data/processed/oscar2026_nominees.json')

# Category key → full name
from src.config import OSCAR_KEY
KEY_TO_CAT = {v: k for k, v in OSCAR_KEY.items()}

print(f'Stats: {len(stats_preds["categories"])} categories')
print(f'ML:    {len(ml_preds["categories"])} categories')
print(f'LLM:   {len(llm_preds["categories"])} categories')
print(f'GD:    {len(gd_preds["categories"])} categories')

Working directory: /Users/mengtingwan/projects/repos/oscar
Stats: 24 categories
ML:    24 categories
LLM:   24 categories
GD:    24 categories


In [2]:
# Cell 2: (Optional) Re-run any method scripts
# Uncomment whichever you want to re-run

# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'src.methods.stats.build_features'], check=True)
# subprocess.run([sys.executable, '-m', 'src.methods.stats.run_stats'], check=True)
# subprocess.run([sys.executable, '-m', 'src.methods.ml_model.run_ml'], check=True)
# subprocess.run([sys.executable, '-m', 'src.methods.goldderby.run_goldderby'], check=True)
print('Skipping re-run (using cached results)')

Skipping re-run (using cached results)


In [ ]:
# Cell 3: Validation comparison table
print(f"{'Method':<15} {'Accuracy':>10}  {'AUC':>8}  {'N cat-yrs':>10}  Notes")
print('-' * 75)

s = stats_valid
print(f"{'Stats':<15} {s['overall_accuracy']:>9.1%}  {s['overall_auc']:>8.3f}  {s['n_cat_years']:>10}  Season award weights")

m = ml_valid
ml_auc_str = f"{m['overall_auc']:.3f}" if m.get('overall_auc') is not None else "    N/A"
print(f"{'ML':<15} {m['overall_accuracy']:>9.1%}  {ml_auc_str:>8}  {m['n_cat_years']:>10}  Elastic-net/GBT, per-category selection")

g = gd_valid
print(f"{'Gold Derby':<15} {g['overall_accuracy']:>9.1%}  {'N/A':>8}  {g['n_cat_years']:>10}  Top-1 picks only (no numeric odds)")

print(f"{'LLM':<15} {'N/A':>9}  {'N/A':>8}  {'N/A':>10}  Independent analysis — no historical validation")

In [ ]:
# Cell 4: Per-category validation breakdown
KEY_CATS = [
    'picture', 'director', 'actor', 'actress', 'supp_actor', 'supp_actress',
    'original_screenplay', 'adapted_screenplay', 'cinematography', 'editing',
    'production_design', 'costume', 'sound', 'makeup', 'score', 'song',
    'visual_effects', 'documentary', 'animated', 'international',
]

rows = []
for ck in KEY_CATS:
    s_cat = stats_valid.get('by_category', {}).get(ck, {})
    m_cat = ml_valid.get('by_category', {}).get(ck, {})
    g_cat = gd_valid.get('by_category', {}).get(ck, {})
    rows.append({
        'Category': ck,
        'Stats Acc': f"{s_cat.get('accuracy', 0):.0%}" if s_cat else 'N/A',
        'Stats AUC': f"{s_cat.get('auc', 0):.3f}" if s_cat and s_cat.get('auc') else 'N/A',
        'ML Acc': f"{m_cat.get('accuracy', 0):.0%}" if m_cat and m_cat.get('accuracy') is not None else 'N/A',
        'ML AUC': f"{m_cat.get('loo_auc', 0):.3f}" if m_cat and m_cat.get('loo_auc') else 'N/A',
        'ML Model': m_cat.get('best_model', 'N/A') if m_cat else 'N/A',
        'GD Acc': f"{g_cat.get('accuracy', 0):.0%}" if g_cat else 'N/A',
    })

df_val = pd.DataFrame(rows)
df_val.set_index('Category', inplace=True)
print(df_val.to_string())

In [5]:
# Cell 5: Helper — look up probabilities from each method for a given category

def get_probs(method_data, cat_name):
    """Return {nominee_name: probability} dict for a given category name."""
    cats = method_data.get('categories', {})
    entries = cats.get(cat_name, [])
    if not entries:
        return {}
    return {e['name']: e['probability'] for e in entries}

print('Helper loaded. Example — Best Picture probabilities:')
cat = 'Best Motion Picture of the Year'
for method_name, data in [('Stats', stats_preds), ('ML', ml_preds), ('LLM', llm_preds), ('GD', gd_preds)]:
    probs = get_probs(data, cat)
    if probs:
        top = max(probs, key=probs.get)
        print(f'  {method_name:<8}: {top[:35]} ({probs[top]:.0%})')

Helper loaded. Example — Best Picture probabilities:
  Stats   : One Battle After Another (60%)
  ML      : One Battle After Another (61%)
  LLM     : One Battle After Another (55%)
  GD      : One Battle After Another (76%)


In [ ]:
# ============================================================
# Cell 6: ENSEMBLE WEIGHTS  ← Edit these to recompute!
# ============================================================
#
# Weights will be normalized to sum to 1.0.
# Set a weight to 0 to exclude that method.
#
# Validation performance (2020-2025):
#   Stats:      72.5% acc, 0.786 AUC
#   ML:         73.2% acc, 0.816 AUC (elastic-net/GBT, per-category selection)
#   Gold Derby: 89.6% acc (top-1 picks; 96 cat-years)
#   LLM:        N/A (independent 2026 analysis only)

W_STATS = 0.25
W_ML    = 0.25
W_LLM   = 0.25
W_GD    = 0.25

print(f'Ensemble weights (before normalization):')
print(f'  Stats: {W_STATS}, ML: {W_ML}, LLM: {W_LLM}, GD: {W_GD}')
w_total = W_STATS + W_ML + W_LLM + W_GD
w_stats_n = W_STATS / w_total
w_ml_n    = W_ML    / w_total
w_llm_n   = W_LLM   / w_total
w_gd_n    = W_GD    / w_total
print(f'Normalized: Stats={w_stats_n:.2f}, ML={w_ml_n:.2f}, LLM={w_llm_n:.2f}, GD={w_gd_n:.2f}')

In [7]:
# Cell 7: Compute ensemble predictions for all categories

def ensemble_category(cat_name, nominees_list):
    """Return list of {name, ensemble_prob, stats_prob, ml_prob, llm_prob, gd_prob} dicts."""
    # Get probabilities from each method
    s_probs  = get_probs(stats_preds, cat_name)
    m_probs  = get_probs(ml_preds,    cat_name)
    l_probs  = get_probs(llm_preds,   cat_name)
    g_probs  = get_probs(gd_preds,    cat_name)

    # Nominee names from the official nominees JSON
    nom_names = [', '.join(n.get('primaryNames', [])) for n in nominees_list]
    n = len(nom_names)

    def lookup(probs_dict, name):
        """Fuzzy lookup — try exact first, then substring."""
        if name in probs_dict:
            return probs_dict[name]
        for k, v in probs_dict.items():
            if name.lower() in k.lower() or k.lower() in name.lower():
                return v
        return 1.0 / n if n > 0 else 0.0

    results = []
    for name in nom_names:
        sp = lookup(s_probs,  name) if s_probs  else 1.0 / n
        mp = lookup(m_probs,  name) if m_probs  else 1.0 / n
        lp = lookup(l_probs,  name) if l_probs  else 1.0 / n
        gp = lookup(g_probs,  name) if g_probs  else 1.0 / n

        ens = w_stats_n * sp + w_ml_n * mp + w_llm_n * lp + w_gd_n * gp
        results.append({
            'name': name,
            'ensemble': round(ens, 4),
            'stats': round(sp, 4),
            'ml': round(mp, 4),
            'llm': round(lp, 4),
            'gd': round(gp, 4),
        })

    # Normalize ensemble
    total_ens = sum(r['ensemble'] for r in results)
    if total_ens > 0:
        for r in results:
            r['ensemble'] = round(r['ensemble'] / total_ens, 4)

    results.sort(key=lambda x: -x['ensemble'])
    return results


# Build ensemble for all categories
ensemble_all = {}
for cat_data in nominees_raw:
    cat_name = cat_data['category']
    ensemble_all[cat_name] = ensemble_category(cat_name, cat_data['nominations'])

print(f'Ensemble computed for {len(ensemble_all)} categories')

Ensemble computed for 24 categories


In [8]:
# Cell 8: Print ensemble predictions — all categories

FOCUS_CATS = [
    'Best Motion Picture of the Year',
    'Best Achievement in Directing',
    'Best Performance by an Actor in a Leading Role',
    'Best Performance by an Actress in a Leading Role',
    'Best Performance by an Actor in a Supporting Role',
    'Best Performance by an Actress in a Supporting Role',
    'Best Original Screenplay',
    'Best Adapted Screenplay',
    'Best Achievement in Cinematography',
    'Best Achievement in Film Editing',
    'Best Achievement in Production Design',
    'Best Achievement in Costume Design',
    'Best Sound',
    'Best Achievement in Makeup and Hairstyling',
    'Best Achievement in Music Written for Motion Pictures (Original Score)',
    'Best Achievement in Music Written for Motion Pictures (Original Song)',
    'Best Achievement in Visual Effects',
    'Best Documentary Feature',
    'Best Animated Feature Film',
    'Best Animated Short Film',
    'Best Live Action Short Film',
    'Best Documentary Short Film',
    'Best International Feature Film',
    'Best Casting',
]

print(f"{'Category':<55} {'Predicted Winner':<32} {'Ens':>5}  {'Stat':>5}  {'ML':>5}  {'LLM':>5}  {'GD':>5}")
print('-' * 120)
for cat in FOCUS_CATS:
    if cat not in ensemble_all:
        continue
    top = ensemble_all[cat][0]
    print(f"{cat[:55]:<55} {top['name'][:32]:<32} {top['ensemble']:>4.0%}  {top['stats']:>4.0%}  {top['ml']:>4.0%}  {top['llm']:>4.0%}  {top['gd']:>4.0%}")

Category                                                Predicted Winner                   Ens   Stat     ML    LLM     GD
------------------------------------------------------------------------------------------------------------------------
Best Motion Picture of the Year                         One Battle After Another          63%   60%   61%   55%   76%
Best Achievement in Directing                           Paul Thomas Anderson              85%   75%   98%   75%   94%
Best Performance by an Actor in a Leading Role          Michael B. Jordan                 54%   32%   76%   65%   43%
Best Performance by an Actress in a Leading Role        Jessie Buckley                    86%   59%   95%   96%   96%
Best Performance by an Actor in a Supporting Role       Sean Penn                         47%   33%   42%   50%   63%
Best Performance by an Actress in a Supporting Role     Amy Madigan                       45%   36%   52%   45%   47%
Best Original Screenplay                        

In [9]:
# Cell 9: Detailed breakdown for contested categories

CONTESTED = [
    'Best Motion Picture of the Year',
    'Best Performance by an Actor in a Leading Role',
    'Best Performance by an Actor in a Supporting Role',
    'Best Performance by an Actress in a Supporting Role',
    'Best Achievement in Film Editing',
    'Best Achievement in Production Design',
    'Best Documentary Feature',
    'Best International Feature Film',
    'Best Casting',
]

for cat in CONTESTED:
    if cat not in ensemble_all:
        continue
    print(f'\n{cat}')
    print(f"  {'Name':<35} {'Ens':>5}  {'Stat':>5}  {'ML':>5}  {'LLM':>5}  {'GD':>5}")
    print('  ' + '-' * 65)
    for r in ensemble_all[cat]:
        bar = '█' * int(r['ensemble'] * 20)
        print(f"  {r['name'][:35]:<35} {r['ensemble']:>4.0%}  {r['stats']:>4.0%}  {r['ml']:>4.0%}  {r['llm']:>4.0%}  {r['gd']:>4.0%}  {bar}")


Best Motion Picture of the Year
  Name                                  Ens   Stat     ML    LLM     GD
  -----------------------------------------------------------------
  One Battle After Another             63%   60%   61%   55%   76%  ████████████
  Sinners                              25%    3%   34%   40%   20%  ████
  Sentimental Value                     3%    9%    2%    1%    0%  
  Hamnet                                2%    5%    0%    2%    1%  
  The Secret Agent                      2%    7%    1%    0%    1%  
  Frankenstein                          1%    3%    1%    0%    0%  
  Train Dreams                          1%    3%    0%    1%    1%  
  Marty Supreme                         1%    3%    0%    0%    0%  
  Bugonia                               1%    3%    0%    0%    0%  
  F1: The Movie                         1%    3%    0%    0%    0%  

Best Performance by an Actor in a Leading Role
  Name                                  Ens   Stat     ML    LLM     GD
 

In [ ]:
# Cell 10: (Optional) Save final ensemble predictions
# Uncomment to save to results/final/

# os.makedirs('results/final', exist_ok=True)
# 
# final_preds = {
#     'method': 'ensemble',
#     'year': 2026,
#     'ensemble_weights': {
#         'stats': round(w_stats_n, 3),
#         'ml': round(w_ml_n, 3),
#         'llm': round(w_llm_n, 3),
#         'goldderby': round(w_gd_n, 3),
#     },
#     'categories': {}
# }
# 
# for cat_name, ens_list in ensemble_all.items():
#     llm_cat = {e['name']: e for e in llm_preds['categories'].get(cat_name, [])}
#     stats_cat = {e['name']: e for e in stats_preds['categories'].get(cat_name, [])}
#     final_preds['categories'][cat_name] = []
#     for r in ens_list:
#         entry = {
#             'name': r['name'],
#             'probability': r['ensemble'],
#             'probabilities_by_method': {
#                 'ensemble': r['ensemble'],
#                 'stats': r['stats'],
#                 'ml': r['ml'],
#                 'llm': r['llm'],
#                 'goldderby': r['gd'],
#             }
#         }
#         llm_entry = llm_cat.get(r['name'])
#         if llm_entry:
#             entry['rationale'] = llm_entry.get('rationale', {})
#         stats_entry = stats_cat.get(r['name'])
#         if stats_entry:
#             entry['season_wins'] = stats_entry.get('rationale', {}).get('season_wins', [])
#         final_preds['categories'][cat_name].append(entry)
# 
# out_path = 'results/final/predictions_2026.json'
# with open(out_path, 'w') as f:
#     json.dump(final_preds, f, indent=2)
# print(f'Saved → {out_path}')

print('(Save cell is commented out — uncomment to persist results/final/predictions_2026.json)')

In [10]:
# Cell 11: Print clean final predictions table

print('=' * 80)
print('98th Academy Awards (March 15, 2026) — Final Ensemble Predictions')
print('=' * 80)
print(f'Weights: Stats={w_stats_n:.0%}, ML={w_ml_n:.0%}, LLM={w_llm_n:.0%}, GD={w_gd_n:.0%}')
print()

SHORT_CATS = {
    'Best Motion Picture of the Year': 'Best Picture',
    'Best Achievement in Directing': 'Director',
    'Best Performance by an Actor in a Leading Role': 'Actor',
    'Best Performance by an Actress in a Leading Role': 'Actress',
    'Best Performance by an Actor in a Supporting Role': 'Supp. Actor',
    'Best Performance by an Actress in a Supporting Role': 'Supp. Actress',
    'Best Original Screenplay': 'Original Screenplay',
    'Best Adapted Screenplay': 'Adapted Screenplay',
    'Best Achievement in Cinematography': 'Cinematography',
    'Best Achievement in Film Editing': 'Film Editing',
    'Best Achievement in Production Design': 'Production Design',
    'Best Achievement in Costume Design': 'Costume Design',
    'Best Sound': 'Sound',
    'Best Achievement in Makeup and Hairstyling': 'Makeup',
    'Best Achievement in Music Written for Motion Pictures (Original Score)': 'Score',
    'Best Achievement in Music Written for Motion Pictures (Original Song)': 'Song',
    'Best Achievement in Visual Effects': 'Visual Effects',
    'Best Documentary Feature': 'Documentary Feature',
    'Best Animated Feature Film': 'Animated Feature',
    'Best Animated Short Film': 'Animated Short',
    'Best Live Action Short Film': 'Live Action Short',
    'Best Documentary Short Film': 'Documentary Short',
    'Best International Feature Film': 'International',
    'Best Casting': 'Casting',
}

print(f"{'Category':<22} {'Predicted Winner':<35} {'Prob':>5}  {'2nd Choice':<28} {'Prob':>5}")
print('-' * 100)
for cat_full, cat_short in SHORT_CATS.items():
    if cat_full not in ensemble_all:
        continue
    ens = ensemble_all[cat_full]
    top1 = ens[0]
    top2 = ens[1] if len(ens) > 1 else None
    second_str = f"{top2['name'][:28]}" if top2 else ''
    second_prob = f"{top2['ensemble']:.0%}" if top2 else ''
    print(f"{cat_short:<22} {top1['name'][:35]:<35} {top1['ensemble']:>4.0%}  {second_str:<28} {second_prob:>5}")

98th Academy Awards (March 15, 2026) — Final Ensemble Predictions
Weights: Stats=25%, ML=25%, LLM=25%, GD=25%

Category               Predicted Winner                     Prob  2nd Choice                    Prob
----------------------------------------------------------------------------------------------------
Best Picture           One Battle After Another             63%  Sinners                        25%
Director               Paul Thomas Anderson                 85%  Ryan Coogler                    8%
Actor                  Michael B. Jordan                    54%  Timothée Chalamet              28%
Actress                Jessie Buckley                       86%  Emma Stone                      4%
Supp. Actor            Sean Penn                            47%  Benicio Del Toro               20%
Supp. Actress          Amy Madigan                          45%  Wunmi Mosaku                   27%
Original Screenplay    Sinners                              67%  Sentimental Value     

In [ ]:
# Cell 12: (Optional) Save validation summary
# Uncomment to persist

# valid_summary = {
#     'stats': {
#         'overall_accuracy': stats_valid['overall_accuracy'],
#         'overall_auc': stats_valid['overall_auc'],
#         'n_cat_years': stats_valid['n_cat_years'],
#         'by_category': stats_valid['by_category'],
#     },
#     'ml_model': {
#         'overall_accuracy': ml_valid['overall_accuracy'],
#         'n_cat_years': ml_valid['n_cat_years'],
#         'note': 'Strict temporal split: train on years < N only',
#         'by_category': ml_valid['by_category'],
#     },
#     'goldderby': {
#         'overall_accuracy': gd_valid['overall_accuracy'],
#         'n_cat_years': gd_valid['n_cat_years'],
#         'note': gd_valid['note'],
#         'by_category': gd_valid['by_category'],
#     },
#     'llm_analysis': {
#         'note': 'Independent analysis — no historical validation (training data leakage prevents 2020-2025 retrospective)'
#     }
# }
# 
# with open('results/final/validation_summary.json', 'w') as f:
#     json.dump(valid_summary, f, indent=2)
# print('Saved → results/final/validation_summary.json')

print('(Save cell is commented out — uncomment to persist results/final/validation_summary.json)')